In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train = pd.read_parquet('/kaggle/input/final-set/train.parquet')
test = pd.read_parquet('/kaggle/input/final-set/test.parquet')
val = pd.read_parquet('/kaggle/input/final-set/val.parquet')
input_col = ['pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'year',
       'hour', 'peak_hour', 'abs_lon_diff', 'abs_lat_diff', 'haversine',
       'jfk_distance', 'lga_distance', 'ewr_distance', 'time_elapsed']
train_input = train[input_col]
train_target = train['fare_amount']
val_input = val[input_col]
val_target = val['fare_amount']
test = test[input_col]

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error


estimators_list = [('model_lr', LinearRegression()),
                  ('model_dt', DecisionTreeRegressor(max_depth =15 , min_samples_leaf = 35, random_state=123)),
                  ('model_rf', RandomForestRegressor(n_jobs = -1,max_depth = 10, n_estimators = 10, random_state=123)),
                  ('model_xgb', XGBRegressor(booster = 'dart', n_jobs = -1, n_estimators = 40, max_depth = 5, gamma = 0.4, eta = 0.315, random_state=123))
                 ]

stacked_model = StackingRegressor(estimators = estimators_list,
                                  final_estimator = LinearRegression(),
                                  n_jobs = -1,
                                  cv = 10,
                                  passthrough = False)

stacked_model.fit(train_input, train_target)
train_pred = stacked_model.predict(train_input)
val_pred = stacked_model.predict(val_input)
train_rmse = np.sqrt(mean_squared_error(train_target,train_pred))
val_rmse = np.sqrt(mean_squared_error(val_target,val_pred))
print(f'trian_rmse: {train_rmse}, val_rmse: {val_rmse}')

test_pred_stacked = stacked_model.predict(test)
submission_stacked = pd.read_csv('/kaggle/input/final-set/sample_submission.csv')
submission_stacked['fare_amount'] = test_pred_stacked
submission_stacked.to_csv('stacking_model_with_lr.csv', index=None)